# Исследование данных и построение моделей машинного обучения для прогнозирования стоимости недвижимости

Данный блокнот содержит полный цикл группового исследования рынка недвижимости портала Zingat:
1. **Этап 1: Исследовательский анализ данных (EDA)** — визуализация структуры датасета и распределений (6 графиков).
2. **Этап 2: Предобработка данных и Feature Engineering** — очистка данных, извлечение новых признаков, обработка пропусков и кодирование.
3. **Этап 3: Построение и обучение 3 моделей регрессии** — Ridge (линейная), Decision Tree (дерево решений), Random Forest (ансамбль бэггинга).
4. **Этап 4: Сравнение качества моделей и выбор финального пайплайна** — диагностические графики (остатки, важность признаков) и сохранение в best_model.pkl.


In [ ]:
import os
import sys
import json
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import Image, display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score


## Этап 1. Исследование данных (EDA)

Загрузка массива данных и предварительный анализ размерности и структуры признаков.


In [ ]:
data_path = '../data/real_estate_data.csv'
df_raw = pd.read_csv(data_path, nrows=100000, low_memory=False)
print(f'Загружено строк: {df_raw.shape[0]}, столбцов: {df_raw.shape[1]}')
display(df_raw.head(3))


### Диаграмма 1. Распределение целевой переменной (цены в TRY)


In [ ]:
display(Image(filename='../reports_and_charts/eda_1_price_distribution.png'))


### Диаграмма 2. Зависимость стоимости от площади недвижимости


In [ ]:
display(Image(filename='../reports_and_charts/eda_2_size_vs_price.png'))


### Диаграмма 3. Разброс цен по типам недвижимости


In [ ]:
display(Image(filename='../reports_and_charts/eda_3_boxplot_subtypes.png'))


### Диаграмма 4. Тепловая карта корреляций числовых характеристик


In [ ]:
display(Image(filename='../reports_and_charts/eda_4_correlation_heatmap.png'))


### Диаграмма 5. Средняя стоимость объектов по ведущим городам


In [ ]:
display(Image(filename='../reports_and_charts/eda_5_bar_top_cities.png'))


### Диаграмма 6. Анализ доли пропущенных значений по признакам


In [ ]:
display(Image(filename='../reports_and_charts/eda_6_missing_values.png'))


## Этап 2. Предобработка данных и Feature Engineering

Очистка аномалий, конвертация валют в TRY, извлечение инженерных признаков:
- `building_age_num`: парсинг строковых диапазонов возраста ("6-10 arası" -> 8.0 лет)
- `room_total`, `rooms`, `living_rooms`: декомпозиция нотации комнат
- `floor_ratio`: относительная высота этажа
- `is_top_floor`, `is_ground_floor`: флаги крайних этажей
- `size_per_room`: средняя площадь комнаты


In [ ]:
sys.path.append('../app')
from estate_utils import extract_features, RATES

df_clean = df_raw.dropna(subset=['price', 'price_currency']).copy()
df_clean['price_try'] = df_clean['price'] * df_clean['price_currency'].map(RATES).fillna(1.0)
df_clean = df_clean[(df_clean['price_try'] >= 100) & (df_clean['price_try'] <= 500_000_000)]
df_clean = df_clean[(df_clean['size'].isna()) | ((df_clean['size'] >= 1) & (df_clean['size'] <= 5000))]

df_feat = extract_features(df_clean)

NUM_COLS = [
    'listing_type', 'tom', 'size', 'building_age_num', 'floor_no_num',
    'total_floor_count_num', 'room_total', 'rooms', 'living_rooms',
    'floor_ratio', 'is_top_floor', 'is_ground_floor', 'size_per_room'
]
CAT_COLS = ['sub_type', 'heating_type', 'city', 'district']

X = df_feat[NUM_COLS + CAT_COLS]
y = df_clean['price_try']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Обучающая выборка: {X_train.shape}, Тестовая выборка: {X_test.shape}')


In [ ]:
num_transformer = Pipeline([('imputer', SimpleImputer(strategy='median'))])
num_transformer_scaled = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False, min_frequency=50))
])

pre_tree = ColumnTransformer([('num', num_transformer, NUM_COLS), ('cat', cat_transformer, CAT_COLS)])
pre_linear = ColumnTransformer([('num', num_transformer_scaled, NUM_COLS), ('cat', cat_transformer, CAT_COLS)])


## Этап 3. Обучение 3 моделей регрессии и оценка метрик

Обучаются 3 модели разной архитектуры:
1. Линейная модель с L2-регуляризацией (Ridge)
2. Одиночное дерево решений (Decision Tree Regressor)
3. Случайный лес (Random Forest Regressor — Ансамбль бэггинга)


In [ ]:
models = {
    'Линейная модель (Ridge)': Pipeline([('pre', pre_linear), ('model', Ridge(alpha=100.0, random_state=42))]),
    'Дерево решений (Decision Tree)': Pipeline([('pre', pre_tree), ('model', DecisionTreeRegressor(max_depth=12, min_samples_leaf=20, random_state=42))]),
    'Случайный лес (Random Forest)': Pipeline([('pre', pre_tree), ('model', RandomForestRegressor(n_estimators=60, max_depth=14, min_samples_leaf=10, n_jobs=-1, random_state=42))])
}

trained_models = {}
results_list = []

for name, pipe in models.items():
    t0 = time.time()
    pipe.fit(X_train, y_train)
    dur = round(time.time() - t0, 2)
    trained_models[name] = pipe
    preds = np.clip(pipe.predict(X_test), 0, None)
    mae = round(float(mean_absolute_error(y_test, preds)), 2)
    acc = round(float(r2_score(y_test, preds)), 4)
    results_list.append({
        'Модель': name,
        'MAE (TRY)': mae,
        'R²': acc,
        'Время обучения (с)': dur
    })
    print(f'Обучена: {name} за {dur} сек.')


### Сводная таблица сравнения моделей


In [ ]:
df_comparison = pd.DataFrame(results_list)
display(df_comparison)


### Диаграмма 7. Сравнение метрик качества моделей


In [ ]:
display(Image(filename='../reports_and_charts/model_comparison_metrics.png'))


## Этап 4. Выбор финальной модели и диагностика

Лучшие показатели качества продемонстрировал ансамбль **Random Forest**.
Ниже приведены диагностические графики качества работы финальной модели.


### Диаграмма 8. Фактические цены против спрогнозированных (Random Forest)


In [ ]:
display(Image(filename='../reports_and_charts/actual_vs_predicted_best.png'))


### Диаграмма 9. Анализ остатков модели


In [ ]:
display(Image(filename='../reports_and_charts/residuals_plot_best.png'))


### Диаграмма 10. Оценка важности признаков


In [ ]:
display(Image(filename='../reports_and_charts/feature_importance_best.png'))


### Сохранение финальной модели в best_model.pkl


In [ ]:
best_pipe = trained_models['Случайный лес (Random Forest)']
joblib.dump(best_pipe, '../data/best_model.pkl')
print('Модель успешно сохранена в data/best_model.pkl')
